In [ ]:
import os
import gdown

MODEL_PATH = "best_model.pth"
FILE_ID = "PASTE_YOUR_ID_HERE" # Just the ID, not the whole link

if not os.path.exists(MODEL_PATH):
    print("🛰️ Fetching model from Drive...")
    url = f'https://drive.google.com/uc?id={FILE_ID}'
    gdown.download(url, MODEL_PATH, quiet=False)

In [ ]:
# --- RUN THIS FIRST TO SYNC GITHUB FILES ---
import os
import sys

# Clone and enter the folder (Change USER/REPO to yours)
if not os.path.exists('/content/defect-detection'):
    !git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git /content/temp_repo
    !cp -r /content/temp_repo/defect-detection/* /content/
    !rm -rf /content/temp_repo

sys.path.append('/content/')
!pip install -q streamlit pyngrok torchvision
print(" Project files and dependencies are ready!")

In [ ]:
app_code = '''
import streamlit as st
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

CLASS_NAMES = ["Crazing", "Inclusion", "Normal", "Patches", "Pitted", "Rolled", "Rust", "Scratches"]
# CHANGED: Look for the model in the current local folder
SAVE_PATH   = "best_model.pth"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@st.cache_resource
def load_model():
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 8)
    model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model
# ... (rest of your predict and UI code) ...
'''
with open("app.py", "w") as f:
    f.write(app_code)

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

try:
    # Use the same Secret name you set up for the previous project
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)

    # Start Streamlit in background
    import subprocess
    import time
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

    time.sleep(5)
    public_url = ngrok.connect(8501).public_url
    print(f"🚀 DASHBOARD LIVE AT: {public_url}")
except Exception as e:
    print(f"❌ Error: {e}. Ensure NGROK_TOKEN is in Colab Secrets.")